# Checking if T4 GPU is used (before ssh)

In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


# Installing dependencies

In [7]:
# Install standard, stable versions compatible with the GPU 
%pip install -q ultralytics roboflow torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 138.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 10.8 MB/s eta 0:00:00


# Download Vietnamese License Plate Dataset from Roboflow

In [8]:
from roboflow import Roboflow
rf = Roboflow(api_key="roboflow api key")  # Replace with your actual Roboflow API key
# https://app.roboflow.com/alphajack2006-gmail-com/vietnamese-car-license-plate-vlbzz/train
# location of the dataset in Roboflow.
project = rf.workspace("alphajack2006-gmail-com").project("vietnamese-car-license-plate-vlbzz")
version = project.version(1)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Vietnamese-Car-License-Plate--1 in yolov8:: 100%|██████████| 16513/16513 [00:01<00:00, 8971.79it/s] 


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


Applied Fine Tuning (leveraging a pretrained model to specialise it in finding license plates)

I did not use any optimisation paths like: adding shears or freezing any base layers.


# Train YOLOv8 Model

In [9]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,       # Full convergence for bounding box regression
    imgsz=640,       # High resolution so small 2-line plates do not blur
    batch=16,
    patience=5,     # Prevents cutting off early
    name="vietnam_plate_lpr",
    save=True
)

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Vietnamese-Car-License-Plate--1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vietnam_plate_lpr, nbs=64, nms=None, op

also, on the side note: mp950-95 shows how tight the bounding boxes are fit around the license plate. since we're doing OCR within a bounding box, i'd need to manually expand the bounding boxes before pushing it to an OCR model that will extract the text.

# Download Trained best.pt Weights to Your Local Machine

In [10]:
from google.colab import files
best = YOLO("runs/detect/vietnam_plate_lpr/weights/best.pt")
best.export(format="onnx", imgsz=640, simplify=True)

files.download("runs/detect/vietnam_plate_lpr/weights/best.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model is downloaded to colab notebook's local runtime; please re-download best.pt from there and replace it in your directory's Models_yolo, if retrained.


If on VSCODE's colab extension .ipynb (what I used), go to the colab extension, click on the existing remote server, and find /runs/... to download the .pt model.